In [ ]:
import transformers
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# Underthe hood uses tensors 
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device)



In [ ]:
text = "Oh great, another broken screen. Just what I always wanted"

inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model(**inputs)
# logits
print(outputs)

# predicted class
predicted_class = outputs.logits.argmax(dim=-1).item()
print(model.config.id2label[predicted_class])


## Text Generating and Streaming

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM,  TextStreamer
import torch

# check gpu compatibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

# load tokenizer and model
model_id = "Qwen/Qwen3-0.6B" # MNN-Pocket-0.3B
tokenizer = AutoTokenizer.from_pretrained(model_id, return_tensors="pt")
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=True)

In [ ]:
# Text Generation
text = """
write a story on rich people
"""
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    # Probablistic sampling instead of greedy_sampling
    outputs = model.generate(
                                **inputs,
                                temperature=0.7,
                                do_sample=True,
                                max_new_tokens=200,
                                top_p=0.9,
                                repetition_penalty=1.4,
                            )
res = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(res) 

In [ ]:
# Streaming
prompt = "what is 5 plu 8."
inputs = tokenizer(
    prompt,
    return_tensors = "pt"
)

model.eval()
with torch.inference_mode():
    model.generate(
        **inputs,
        top_p=0.9,
        repetition_penalty=1.3,
        temperature=.7,
        max_new_tokens=500,
        do_sample=True,
        streamer=streamer,
        eos_token_id=tokenizer.eos_token_id
    )

In [ ]:
# Embeddings 
from sentence_transformers import SentenceTransformer

model_id = "Qwen/Qwen3-Embedding-0.6B"
embedding_model = SentenceTransformer(
    model_id,
    device=device
)

In [ ]:
queries = [
    "place"
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

query_embedding = embedding_model.encode(queries, prompt_name="query")
document_embedding = embedding_model.encode(documents)

similarity = embedding_model.similarity(query_embedding, document_embedding)

ranked = sorted(zip(similarity[0].tolist() , documents), reverse=True)
for score, doc in ranked:
    print(f"[Score: {score:.4f}] {doc}")

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    'text-generation',
    model="openai-community/gpt2",
    device=torch.device("cuda")
)



Loading weights: 100%|██████████| 148/148 [00:00<00:00, 184.93it/s]


ValueError: Input length of input_ids is 200, but `max_length` is set to 200. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.

In [23]:
text = """ New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving court, Barrientos was arrested and charged with theft of service and criminal trespass for allegedly sneaking into the New York subway through an emergency exit, said Detective
Annette Markowski, a police spokeswoman. In total, Barrientos has been married 10 times, with nine of her marriages occurring between 1999 and 2002.
All occurred either in Westchester County, Long Island, New Jersey or the Bronx. She is believed to still be married to four men, and at one time, she was married to eight men at once, prosecutors say.
Prosecutors said the immigration scam involved some of her husbands, who filed for permanent residence status shortly after the marriages.
Any divorces happened only after such filings were approved. It was unclear whether any of the men will be prosecuted.
The case was referred to the Bronx District Attorney\'s Office by Immigration and Customs Enforcement and the Department of Homeland Security\'s
Investigation Division. Seven of the men are from so-called "red-flagged" countries, including Egypt, Turkey, Georgia, Pakistan and Mali.
Her eighth husband, Rashid Rajput, was deported in 2006 to his native Pakistan after an investigation by the Joint Terrorism Task Force.
If convicted, Barrientos faces up to four years in prison.  Her next court appearance is scheduled for May 18.
"""

res = summarizer(
    f'SUMMARIZE{text}',
    max_new_tokens = 200,
    min_length = 50,
    do_sample = False
)

In [ ]:
res[-1]['generated_text']

'SUMMARIZE New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.\nA year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.\nOnly 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.\nIn 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.\nBarrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the\n2010 marriage license application, according to court documents.\nProsecutors said the marriages were part of an immigration scam.\nOn Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.\nAfter leaving